# v0.2 · Notebook 01: Collect sources

**LLM Security Gateway** · builds the raw material for the v0.2 multilingual prompt-injection dataset.

What this notebook does:
1. Downloads 5 openly licensed sources (attacks, indirect attacks, harmless text in English and Spanish)
2. Converts them to one schema: `text, label, source, lang, family, origin, license, split_hint`
3. Detects languages, removes duplicates and texts labelled both ways
4. Shows what's inside, so we can decide the next steps together
5. Saves `sources_v0.2.parquet` for notebook 02 (splits + Spanish expansion)

Runtime: **CPU is enough** (no GPU needed). Takes about 3–5 minutes.

| Source | Licence | Used for |
|---|---|---|
| deepset/prompt-injections | Apache-2.0 | direct attacks + harmless |
| Lakera/gandalf_ignore_instructions | MIT | direct attacks (real players) |
| jackhhao/jailbreak-classification | Apache-2.0 | role-play jailbreaks + harmless |
| microsoft/BIPIA (email part) | MIT | **indirect** attacks hidden in emails |
| argilla/databricks-dolly-15k-curated-multilingual | CC BY-SA 3.0 | harmless questions + documents, EN + ES |
| This repo: `demo_traffic.jsonl`, `my_attacks.jsonl` | project | **evaluation only**, never trained on |

Excluded: `xTRam1/safe-guard-prompt-injection` has no licence, so we have no permission to redistribute it.

In [ ]:
!pip -q install datasets langid

In [ ]:
import json, random, re, unicodedata, collections
import pandas as pd, requests, langid
from datasets import load_dataset

SEED = 42
rng = random.Random(SEED)
OUT_DIR = "/content/v02"
REPO_RAW = "https://raw.githubusercontent.com/edithngalame/llm-security-gateway/main"
BIPIA_RAW = "https://raw.githubusercontent.com/microsoft/BIPIA/main/benchmark"

# Size caps keep any single source from dominating the dataset
CAPS = {
    "bipia_train": 1500,        # emails x train attacks
    "bipia_test": 500,          # emails x held-out attacks (never seen in training)
    "dolly_instr_per_lang": 3000,
    "dolly_ctx_per_lang": 1500,
    "dolly_ctx_chars": 1200,    # documents are trimmed to a retrieval-sized chunk
}
records = []
pd.set_option("display.max_colwidth", 120)

## Helper functions
The same functions were unit-tested before this notebook was written.

In [ ]:
SCHEMA = ["text", "label", "source", "lang", "family", "origin", "license", "split_hint"]


def rec(text, label, source, lang, family, origin, license_, split_hint="pool"):
    return {"text": text, "label": int(label), "source": source, "lang": lang,
            "family": family, "origin": origin, "license": license_, "split_hint": split_hint}


def dedup_key(text: str) -> str:
    """Case-, accent-width- and punctuation-insensitive key for finding duplicates."""
    t = unicodedata.normalize("NFKC", text).lower()
    t = re.sub(r"[^\w]+", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def insert_attack(context: str, attack: str, position: str, rng: random.Random) -> str:
    """Hide an attack instruction inside a document (BIPIA-style indirect injection)."""
    if position == "start":
        return f"{attack}\n{context}"
    if position == "end":
        return f"{context}\n{attack}"
    sentences = re.split(r"(?<=[.!?])\s+", context)
    if len(sentences) < 2:
        return f"{context}\n{attack}"
    cut = rng.randint(1, len(sentences) - 1)
    return " ".join(sentences[:cut]) + f" {attack} " + " ".join(sentences[cut:])


def build_bipia(contexts, attacks, cap, rng, split_hint):
    """contexts: list of str. attacks: dict category -> list[str]. Returns list of records."""
    flat = [(cat, a) for cat, items in attacks.items() for a in items]
    combos = [(c, cat, a) for c in contexts for cat, a in flat]
    rng.shuffle(combos)
    out = []
    for c, cat, a in combos[:cap]:
        pos = rng.choice(["start", "middle", "end"])
        fam = "bipia_" + re.sub(r"[^a-z0-9]+", "_", cat.lower()).strip("_")
        out.append(rec(insert_attack(c, a, pos, rng), 1, "retrieved", "en", fam,
                       "microsoft/BIPIA", "MIT", split_hint))
    return out


def clean(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Strip, drop empties, remove exact/near duplicates, drop texts labelled both ways."""
    report = {"input": len(df)}
    df = df.copy()
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"].str.len() >= 3]
    report["after_empty"] = len(df)
    df["key"] = df["text"].map(dedup_key)
    labels_per_key = df.groupby("key")["label"].nunique()
    conflicting = set(labels_per_key[labels_per_key > 1].index)
    report["conflicting_texts"] = len(conflicting)
    df = df[~df["key"].isin(conflicting)]
    # keep the first copy; prefer eval-only copies so hand-written tests stay intact
    df = df.assign(_prio=(df["split_hint"] != "eval_only").astype(int)).sort_values("_prio", kind="stable")
    df = df.drop_duplicates("key", keep="first").drop(columns="_prio")
    report["after_dedup"] = len(df)
    return df.reset_index(drop=True), report


def read_jsonl_text(raw: str):
    return [json.loads(line) for line in raw.splitlines()
            if line.strip() and not line.lstrip().startswith("//")]

## 1. deepset/prompt-injections

In [ ]:
ds = load_dataset("deepset/prompt-injections")
n0 = len(records)
for split in ds:
    for r in ds[split]:
        fam = "deepset_injection" if r["label"] == 1 else "deepset_benign"
        records.append(rec(r["text"], r["label"], "user", None, fam, "deepset/prompt-injections", "Apache-2.0"))
print(f"added {len(records) - n0} rows")

## 2. Lakera/gandalf_ignore_instructions
Every row is a real attack typed by players trying to make the Gandalf game reveal its password.

In [ ]:
ds = load_dataset("Lakera/gandalf_ignore_instructions")
n0 = len(records)
for split in ds:
    for r in ds[split]:
        records.append(rec(r["text"], 1, "user", None, "gandalf", "Lakera/gandalf_ignore_instructions", "MIT"))
print(f"added {len(records) - n0} rows")

## 3. jackhhao/jailbreak-classification

In [ ]:
ds = load_dataset("jackhhao/jailbreak-classification")
n0 = len(records)
for split in ds:
    for r in ds[split]:
        is_attack = r["type"] == "jailbreak"
        records.append(rec(r["prompt"], int(is_attack), "user", None,
                           "jailbreak_roleplay" if is_attack else "jackhhao_benign",
                           "jackhhao/jailbreak-classification", "Apache-2.0"))
print(f"added {len(records) - n0} rows")

## 4. microsoft/BIPIA: indirect injection in emails

BIPIA ships **separate attack lists for training and testing**. We insert attacks into real emails at the start, middle or end. Emails with *test* attacks are marked `heldout_attack`: they measure whether the model catches attack styles it never saw.

In [ ]:
def get(url):
    r = requests.get(url, timeout=60); r.raise_for_status(); return r.text

attacks_train = json.loads(get(f"{BIPIA_RAW}/text_attack_train.json"))
attacks_test = json.loads(get(f"{BIPIA_RAW}/text_attack_test.json"))
emails_train = [row["context"] for row in read_jsonl_text(get(f"{BIPIA_RAW}/email/train.jsonl"))]
emails_test = [row["context"] for row in read_jsonl_text(get(f"{BIPIA_RAW}/email/test.jsonl"))]
print(f"attack categories: {len(attacks_train)} | train attacks: {sum(map(len, attacks_train.values()))} "
      f"| test attacks: {sum(map(len, attacks_test.values()))} | emails: {len(emails_train)} train, {len(emails_test)} test")

n0 = len(records)
records += build_bipia(emails_train, attacks_train, CAPS["bipia_train"], rng, "pool")
records += build_bipia(emails_test, attacks_test, CAPS["bipia_test"], rng, "heldout_attack")
# the same emails WITHOUT an attack: harmless retrieved documents
for e in emails_train + emails_test:
    records.append(rec(e, 0, "retrieved", "en", "bipia_clean_email", "microsoft/BIPIA", "MIT"))
print(f"added {len(records) - n0} rows")

## 5. Dolly-15k multilingual: harmless questions and documents (EN + ES)

The Spanish half is **machine-translated** from English. Fine for harmless examples, but we'll note it in the dataset card, and notebook 02 adds natively written Spanish.

In [ ]:
n0 = len(records)
for lang in ["en", "es"]:
    ds = load_dataset("argilla/databricks-dolly-15k-curated-multilingual", split=lang).shuffle(seed=SEED)
    origin = "argilla/databricks-dolly-15k-curated-multilingual" + (" (machine-translated)" if lang == "es" else "")
    for r in ds.select(range(min(CAPS["dolly_instr_per_lang"], len(ds)))):
        records.append(rec(r["instruction"], 0, "user", lang, "dolly_instruction", origin, "CC-BY-SA-3.0"))
    ctx = [r["context"] for r in ds if r["context"] and len(r["context"]) > 200][:CAPS["dolly_ctx_per_lang"]]
    for c in ctx:
        records.append(rec(c[:CAPS["dolly_ctx_chars"]], 0, "retrieved", lang, "dolly_context", origin, "CC-BY-SA-3.0"))
print(f"added {len(records) - n0} rows")

## 6. Your hand-written sets from GitHub: evaluation only
These are never used for training, so they stay a fair test.

In [ ]:
n0 = len(records)
for name in ["demo_traffic.jsonl", "my_attacks.jsonl"]:
    try:
        rows = read_jsonl_text(get(f"{REPO_RAW}/data/samples/{name}"))
    except Exception as e:
        print(f"could not load {name}: {e}"); continue
    for r in rows:
        records.append(rec(r["text"], r["label"], r.get("source", "user"), r.get("lang"),
                           r.get("family", "handwritten"), f"handwritten/{name}", "project", "eval_only"))
print(f"added {len(records) - n0} rows")

## 7. Language detection + cleaning

- Detects the language where the source didn't say (limited to 6 likely languages to reduce mistakes)
- English and Spanish go to the main pool. Other languages (German in deepset, for example) are kept aside as a **cross-language test**, like your Italian attack
- Removes duplicates, and texts that appear with **both** labels in different sources

In [ ]:
df = pd.DataFrame(records, columns=SCHEMA)
langid.set_languages(["en", "es", "de", "fr", "it", "pt"])
missing = df["lang"].isna()
df.loc[missing, "lang"] = df.loc[missing, "text"].map(lambda t: langid.classify(t)[0])

other = ~df["lang"].isin(["en", "es"]) & (df["split_hint"] == "pool")
df.loc[other, "split_hint"] = "other_lang"

df, report = clean(df)
print(json.dumps(report, indent=2))

## 8. What's inside

In [ ]:
print(f"TOTAL: {len(df):,} rows | attacks: {(df.label == 1).sum():,} | harmless: {(df.label == 0).sum():,}\n")
display(pd.crosstab(df["origin"], df["label"], margins=True).rename(columns={0: "harmless", 1: "attack"}))
display(pd.crosstab([df["source"], df["lang"]], df["label"], margins=True).rename(columns={0: "harmless", 1: "attack"}))
display(pd.crosstab(df["split_hint"], df["label"]).rename(columns={0: "harmless", 1: "attack"}))

In [ ]:
# Text length per origin: very different lengths between attacks and harmless text
# would let a model cheat by looking at length instead of meaning.
df["chars"] = df["text"].str.len()
display(df.groupby(["origin", "label"])["chars"].describe(percentiles=[.5]).round(0))

In [ ]:
# Read real examples: 3 attacks and 3 harmless per origin
for (origin, label), g in df.groupby(["origin", "label"]):
    print(f"\n=== {origin} | {'ATTACK' if label else 'harmless'} ===")
    for t in g.sample(min(3, len(g)), random_state=SEED)["text"]:
        print(" -", t[:200].replace("\n", " "))

## 9. Save

In [ ]:
import os, shutil
os.makedirs(OUT_DIR, exist_ok=True)
df.drop(columns=["key", "chars"]).to_parquet(f"{OUT_DIR}/sources_v0.2.parquet", index=False)
summary = {
    "rows": len(df), "attacks": int((df.label == 1).sum()), "harmless": int((df.label == 0).sum()),
    "cleaning": report, "caps": CAPS, "seed": SEED,
    "by_origin": df.groupby("origin").size().to_dict(),
    "licenses": df.groupby("origin")["license"].first().to_dict(),
}
json.dump(summary, open(f"{OUT_DIR}/sources_summary.json", "w"), indent=2)
shutil.make_archive("/content/v02_sources", "zip", OUT_DIR)
print("saved:", os.listdir(OUT_DIR))

from google.colab import files
files.download("/content/v02_sources.zip")